# Laboratori 8 · Ushtrime Shtesë — Floyd-Warshall në kod: rrugët, ciklet negative dhe mbyllja kalimtare

> Laboratori kryesor e animoi Floyd-Warshall-in në tre skenarë dhe e gjurmuat me dorë mbi matricën $4 \times 4$.
> Këtu e shkruani **vetë**, bashkë me rindërtimin e rrugës, zbulimin e cikleve negative dhe dy "kushërinjtë" e tij,
> dhe çdo qelizë `kontrollo(...)` ju tregon nëse janë të sakta. Nuk kërkon Pygame.

| # | Ushtrimi | Çfarë mësoni |
|:-:|---|---|
| 1 | Matrica e peshave | $D^{(0)}$: 0 në diagonale, pesha e harkut, $\infty$ |
| 2 | Floyd-Warshall me hapin tjetër | programimi dinamik mbi $k$, matrica `nxt` |
| 3 | Rruga nga `nxt` | rindërtimi i rrugës përpara |
| 4 | Ciklet negative | diagonalja dhe çiftet pa rrugë më të shkurtër |
| 5 | Mbyllja kalimtare | e njëjta ide mbi vlerat `True` / `False` (Warshall) |
| 6 | Dijkstra nga çdo kulm | të gjitha çiftet ndryshe, dhe kur ia vlen |

## Kujtues teorik

**Paraqitja.** Kulmet janë një listë `L = ["A", "B", ...]`; harqet `(u, v, w)`. Matricat janë lista listash me
indekse sipas pozicionit në `L`: `D[i][j]` është distanca nga `L[i]` te `L[j]`. Pafundësia: `INF = float("inf")`.

**Floyd-Warshall.** Pas fazës $k$, `D[i][j]` është rruga më e shkurtër nga $i$ te $j$ që përdor si ndërmjetës vetëm
kulmet $0, 1, \dots, k$:

$$D[i][j] \leftarrow \min\big(D[i][j],\; D[i][k] + D[k][j]\big)$$

Koha $O(V^3)$, hapësira $O(V^2)$. Funksionon me pesha negative, **pa** cikle negative.

**Matrica e hapit tjetër.** `nxt[i][j]` = kulmi i parë pas $i$-së në rrugën më të shkurtër drejt $j$-së. Kur rruga
përmirësohet përmes $k$-së, hapi i parë drejt $j$-së bëhet hapi i parë drejt $k$-së: `nxt[i][j] = nxt[i][k]`.
Rruga lexohet **përpara**: $i \to$ `nxt[i][j]` $\to \dots \to j$.

**Ciklet negative.** Pas ekzekutimit, $D[v][v] < 0$ atëherë dhe vetëm atëherë kur $v$ shtrihet në një cikël me peshë
negative. Për çdo çift $(i, j)$ ku $i$ arrin një kulm të tillë $v$ dhe $v$ arrin $j$, rruga më e shkurtër **nuk ekziston**.

In [1]:
# ── Mjeti i kontrollit ──────────────────────────────────────────────
# Ekzekutojeni këtë qelizë një herë, në fillim. Pas çdo ushtrimi, qeliza
# "kontrollo(...)" teston funksionin tuaj dhe ju tregon nëse është gati.

def kontrollo(titulli, testet):
    try:
        testet()
    except NotImplementedError:
        print(f"⏳ {titulli}: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.")
    except AssertionError as e:
        print(f"✗ {titulli}: {e}")
    except Exception as e:
        print(f"✗ {titulli}: kodi u ndal me gabim  {type(e).__name__}: {e}")
    else:
        print(f"✓ {titulli}: të gjitha testet kaluan!")


import heapq
import random
from itertools import permutations

INF = float("inf")


def harqet_e_rastit(rng, L, p=0.45, peshat=(1, 9)):
    return [(u, v, rng.randint(*peshat)) for u in L for v in L if u != v and rng.random() < p]


def _rruget_e_thjeshta(L, A, i, j):
    # Me forcë brutale: pesha më e vogël mbi të gjitha shtigjet e thjeshta nga L[i] te L[j] (vetëm për grafe të vegjël).
    w = {}
    for u, v, p in A:
        w[(u, v)] = min(p, w.get((u, v), INF))
    if i == j:
        return 0
    me_e_mira = INF
    mes = [x for k, x in enumerate(L) if k not in (i, j)]
    for n in range(len(mes) + 1):
        for rruga in permutations(mes, n):
            shtegu = [L[i], *rruga, L[j]]
            if all((a, b) in w for a, b in zip(shtegu, shtegu[1:])):
                me_e_mira = min(me_e_mira, sum(w[(a, b)] for a, b in zip(shtegu, shtegu[1:])))
    return me_e_mira


# Matrica e Punës në Klasë të Laboratorit 8 (D⁰ me 4 kulme)
L_KLASA = ["0", "1", "2", "3"]
A_KLASA = [("0", "1", 3), ("0", "3", 7), ("1", "0", 8), ("1", "2", 2), ("2", "0", 5), ("2", "3", 1), ("3", "0", 2)]

## Ushtrimi 1 — Matrica e peshave

**Detyra:** `matrica(L, A)` kthen $D^{(0)}$: `0` në diagonale, pesha e harkut për çdo hark (nëse ka disa harqe
midis të njëjtit çift, më e vogla), dhe `INF` kudo tjetër.

In [2]:
def matrica(L, A):
    # PLOTESO: indeksi i çdo kulmi, pastaj 0 / pesha më e vogël / INF
    raise NotImplementedError

In [3]:
def _testet():
    assert matrica(L_KLASA, A_KLASA) == [[0, 3, INF, 7], [8, 0, 2, INF], [5, INF, 0, 1], [2, INF, INF, 0]], \
        "D⁰ e Punës në Klasë"
    assert matrica(["a", "b"], [("a", "b", 5), ("a", "b", 2)]) == [[0, 2], [INF, 0]], "dy harqe a → b: merret 2"
    assert matrica(["a"], []) == [[0]], "një kulm pa harqe"

kontrollo("Ushtrimi 1 — Matrica e peshave", _testet)

⏳ Ushtrimi 1 — Matrica e peshave: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 2 — Floyd-Warshall me hapin tjetër

**Detyra:** `floyd_warshall(W)` merr matricën $D^{(0)}$ dhe kthen `(D, nxt)`, pa e ndryshuar `W`:
- `D` — distancat më të shkurtra;
- `nxt[i][j]` — indeksi i kulmit të parë pas $i$-së në një rrugë më të shkurtër drejt $j$-së, `i` kur `i == j`,
  dhe `None` kur nuk ka rrugë.

Nisni `nxt[i][j] = j` kur ka hark ($i \ne j$, $W[i][j] < \infty$). Në tre ciklet `for k`, `for i`, `for j`:
kur `D[i][k] + D[k][j] < D[i][j]`, përditësoni të dyja.

In [4]:
def floyd_warshall(W):
    # PLOTESO: kopje e W; nxt fillestar; tre cikle me k jashtë
    raise NotImplementedError

In [5]:
def _testet():
    W = matrica(L_KLASA, A_KLASA)
    kopje = [r[:] for r in W]
    D, nxt = floyd_warshall(W)
    assert W == kopje, "floyd_warshall nuk duhet ta ndryshojë W"
    assert D == [[0, 3, 5, 6], [5, 0, 2, 3], [3, 6, 0, 1], [2, 5, 7, 0]], f"D përfundimtare e Punës në Klasë: {D}"
    assert nxt[3][2] == 0 and nxt[0][3] == 1, "3 → 2 nis me 3 → 0; 0 → 3 nis me 0 → 1 (3 + 2 + 1 = 6 < 7)"
    rng = random.Random(8)
    for _ in range(40):
        L = [chr(97 + i) for i in range(rng.randint(1, 5))]
        A = harqet_e_rastit(rng, L)
        D, _ = floyd_warshall(matrica(L, A))
        for i in range(len(L)):
            for j in range(len(L)):
                assert D[i][j] == _rruget_e_thjeshta(L, A, i, j), f"D[{L[i]}][{L[j]}] nuk është rruga më e shkurtër"

kontrollo("Ushtrimi 2 — Floyd-Warshall", _testet)

⏳ Ushtrimi 2 — Floyd-Warshall: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 3 — Rruga nga `nxt`

**Detyra:** `rruga(nxt, i, j)` kthen listën e indekseve nga `i` te `j`, duke ndjekur `nxt` **përpara**
($i$, `nxt[i][j]`, `nxt[nxt[i][j]][j]`, …), ose `None` kur `nxt[i][j]` është `None`.

In [6]:
def rruga(nxt, i, j):
    # PLOTESO: nga i, hap pas hapi drejt j
    raise NotImplementedError

In [7]:
def _testet():
    W = matrica(L_KLASA, A_KLASA)
    D, nxt = floyd_warshall(W)
    assert rruga(nxt, 0, 0) == [0], "rruga nga një kulm te vetja"
    assert rruga(nxt, 0, 3) == [0, 1, 2, 3], f"0 → 3 me peshë 6: {rruga(nxt, 0, 3)}"
    assert rruga(nxt, 3, 2) == [3, 0, 1, 2], f"3 → 2 me peshë 7: {rruga(nxt, 3, 2)}"
    rng = random.Random(9)
    for _ in range(60):
        L = [chr(97 + i) for i in range(rng.randint(1, 6))]
        W = matrica(L, harqet_e_rastit(rng, L, p=0.35))
        D, nxt = floyd_warshall(W)
        for i in range(len(L)):
            for j in range(len(L)):
                P = rruga(nxt, i, j)
                if D[i][j] == INF:
                    assert P is None, "pa rrugë: None"
                else:
                    assert P[0] == i and P[-1] == j, "rruga nis te i dhe mbaron te j"
                    assert sum(W[a][b] for a, b in zip(P, P[1:])) == D[i][j], "pesha e rrugës duhet të jetë D[i][j]"

kontrollo("Ushtrimi 3 — Rruga", _testet)

⏳ Ushtrimi 3 — Rruga: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 4 — Ciklet negative

**Detyra:**
1. `ne_cikel_negativ(D)` — lista e indekseve $v$ me $D[v][v] < 0$, në rend rritës.
2. `pa_rruge_me_te_shkurter(D)` — bashkësia e çifteve `(i, j)` që nuk kanë rrugë më të shkurtër: ekziston një $v$
   me $D[v][v] < 0$, $D[i][v] < \infty$ dhe $D[v][j] < \infty$.

In [8]:
def ne_cikel_negativ(D):
    # PLOTESO
    raise NotImplementedError


def pa_rruge_me_te_shkurter(D):
    # PLOTESO: çiftet që kalojnë nëpër një kulm të një cikli negativ
    raise NotImplementedError

In [9]:
def _testet():
    # A → B (1), B → C (−3), C → A (1): cikli A → B → C → A ka peshë −1. C → D (2); E është i veçuar.
    L = ["A", "B", "C", "D", "E"]
    D, _ = floyd_warshall(matrica(L, [("A", "B", 1), ("B", "C", -3), ("C", "A", 1), ("C", "D", 2)]))
    assert ne_cikel_negativ(D) == [0, 1, 2], "A, B dhe C janë në ciklin negativ"
    keq = pa_rruge_me_te_shkurter(D)
    assert (0, 3) in keq and (3, 3) not in keq and (4, 4) not in keq, "A → D s'ka rrugë më të shkurtër; D → D dhe E → E kanë"
    assert keq == {(i, j) for i in range(3) for j in range(4)}, "nga A, B, C te A, B, C, D: 12 çifte"
    D, _ = floyd_warshall(matrica(L_KLASA, A_KLASA))
    assert ne_cikel_negativ(D) == [] and pa_rruge_me_te_shkurter(D) == set(), "pa pesha negative, asnjë cikël negativ"

kontrollo("Ushtrimi 4 — Ciklet negative", _testet)

⏳ Ushtrimi 4 — Ciklet negative: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


> **Pyetje:** Në shembullin e kontrollit, $D[A][A]$ del $-1$, por $D[B][B]$ del $-1$ dhe $D[C][C]$ del $-2$. Pse
> numrat e diagonales nuk janë "pesha e ciklit"? Çfarë mat ato pas tre fazave të $k$?

## Ushtrimi 5 — Mbyllja kalimtare (Warshall)

E njëjta ide, mbi pyetjen *a ka rrugë?* në vend të *sa e gjatë?*: `R[i][j]` është `True` nëse ka rrugë nga $i$ te $j$
(me gjatësi 0 ose më shumë, pra edhe `R[i][i] = True`):

$$R[i][j] \leftarrow R[i][j] \;\lor\; \big(R[i][k] \land R[k][j]\big)$$

**Detyra:** `mbyllja(L, A)` kthen matricën `R`.

In [10]:
def mbyllja(L, A):
    # PLOTESO: R nga harqet dhe diagonalja, pastaj tre ciklet me k jashtë
    raise NotImplementedError

In [11]:
def _testet():
    R = mbyllja(["a", "b", "c"], [("a", "b", 1), ("b", "c", 1)])
    assert R == [[True, True, True], [False, True, True], [False, False, True]], "a → b → c: a arrin c"
    rng = random.Random(10)
    for _ in range(60):
        L = [chr(97 + i) for i in range(rng.randint(1, 7))]
        A = harqet_e_rastit(rng, L, p=0.25)
        D, _ = floyd_warshall(matrica(L, A))
        assert mbyllja(L, A) == [[d < INF for d in row] for row in D], "R[i][j] ⟺ D[i][j] < ∞"

kontrollo("Ushtrimi 5 — Mbyllja kalimtare", _testet)

⏳ Ushtrimi 5 — Mbyllja kalimtare: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 6 — Dijkstra nga çdo kulm

Me pesha **jo-negative**, të gjitha çiftet merren edhe me Dijkstra nga çdo kulm: $V \cdot O((V + E)\log V)$.

**Detyra:**
1. `dijkstra(L, A, s)` — lista e distancave nga `L[s]` te çdo kulm (`INF` për të paarritshmit), me `heapq`.
2. `te_gjitha_dijkstra(L, A)` — matrica e distancave, një rresht për çdo burim.

In [12]:
def dijkstra(L, A, s):
    # PLOTESO: dist, heap me (d, kulm), dhe kulmet e tabeluar
    raise NotImplementedError


def te_gjitha_dijkstra(L, A):
    # PLOTESO
    raise NotImplementedError

In [13]:
def _testet():
    assert te_gjitha_dijkstra(L_KLASA, A_KLASA) == floyd_warshall(matrica(L_KLASA, A_KLASA))[0], \
        "Dijkstra nga çdo kulm jep të njëjtën matricë si Floyd-Warshall"
    assert dijkstra(["a", "b", "c"], [("a", "b", 2)], 0) == [0, 2, INF], "c nuk arrihet: INF"
    rng = random.Random(11)
    for _ in range(60):
        L = [chr(97 + i) for i in range(rng.randint(1, 7))]
        A = harqet_e_rastit(rng, L, p=0.3, peshat=(0, 9))
        assert te_gjitha_dijkstra(L, A) == floyd_warshall(matrica(L, A))[0], f"ndryshojnë për {A}"

kontrollo("Ushtrimi 6 — Dijkstra nga çdo kulm", _testet)

⏳ Ushtrimi 6 — Dijkstra nga çdo kulm: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Punë në Klasë

**Problemi 1 — `nxt` me dorë.** Për matricën e Punës në Klasë, shkruani `nxt` fillestare dhe pas çdo faze $k$.
Pas cilës fazë ndryshon për herë të fundit `nxt[3][2]`? Lexoni rrugën $3 \to 2$ nga `nxt` përfundimtare.

**Problemi 2 — Cikli negativ.** Për grafin e Ushtrimit 4, ekzekutoni Floyd-Warshall me dorë dhe shkruani diagonalen
pas çdo faze $k$. Kur bëhet negative për herë të parë? Pse $D$ e kulmit E nuk preket?

**Problemi 3 — Warshall.** Për një digraf me harqet $a \to b$, $b \to c$, $c \to a$, $c \to d$, plotësoni $R$ pas
çdo faze. Cilat kulme arrijnë të gjithë të tjerët? Lidheni me komponentet e lidhura fort të Laboratorit 6.

## Pyetje Reflektuese

1. Pse cikli i jashtëm duhet të jetë ai i $k$-së? Çfarë shkon keq nëse ciklet renditen `for i`, `for j`, `for k`?
   Provojeni në kod mbi një graf ku kjo jep një rezultat të gabuar.
2. Rruga nga `nxt` lexohet përpara, ndërsa nga `prev` i Dijkstrës mbrapsht. Pse? Çfarë ruan secila matricë?
3. Për një graf me $V = 2\,000$ dhe $E = 8\,000$, rreth sa hapa bën Floyd-Warshall, dhe sa Dijkstra nga çdo kulm?
   Po për $E \approx V^2$?
4. Mbyllja kalimtare përgjigjet *"a ka rrugë?"*. Si do ta përdornit për të gjetur komponentet e lidhura fort?